# Multi-Source Visual Search: Vanilla VLM Baseline Benchmark (Google Colab)

Evaluates vanilla Vision-Language Models directly on **STASHBench** multi-image queries.
Uses deterministic candidate pools $N \in \{5, 10, 20, \text{crop}\}$ ordered by `gallery_order_seed42.json`.
Results persist incrementally into Google Drive with timestamped filenames and auto-resume.

In [ ]:
# Cell 1: Mount Google Drive & Configure Paths
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Smart Path Auto-Detection
POSSIBLE_DATASET_DIRS = [
    '/content/drive/MyDrive/STASHBench',
    '/content/drive/MyDrive/data/STASHBench',
    '/content/drive/MyDrive/cvpr_research',
    '/content/drive/MyDrive/cvpr_research/dataset',
    '/content/drive/MyDrive/VisualSearch/STASHBench',
]

DATASET_PATH = None
IMAGE_DIR = None

for d in POSSIBLE_DATASET_DIRS:
    for anno in ['annotations_part1234_merged.json', '200_samples.json', 'anno.json']:
        p = os.path.join(d, anno)
        if os.path.isfile(p):
            DATASET_PATH = p
            break
    if DATASET_PATH:
        for img_sub in ['pool', 'images', '.']:
            ip = os.path.join(d, img_sub)
            if os.path.isdir(ip) and len(os.listdir(ip)) > 0:
                IMAGE_DIR = ip
                break
        break

if not DATASET_PATH:
    DATASET_PATH = '/content/drive/MyDrive/STASHBench/annotations_part1234_merged.json'
if not IMAGE_DIR:
    IMAGE_DIR = '/content/drive/MyDrive/STASHBench/pool'

OUTPUT_DIR = '/content/drive/MyDrive/mssearch_results/baseline'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Dataset Path : {DATASET_PATH} (exists: {os.path.isfile(DATASET_PATH)})')
print(f'Image Dir    : {IMAGE_DIR} (exists: {os.path.isdir(IMAGE_DIR)})')
print(f'Output Dir   : {OUTPUT_DIR}')

In [ ]:
# Cell 2: GPU Sanity Check & Clone/Update Repository
!nvidia-smi

REPO_DIR = '/content/MSSearch'
BRANCH = 'draft/improve_all'

GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or userdata.get('GH_TOKEN')
    hf_tok = userdata.get('HF_TOKEN')
    if hf_tok:
        os.environ['HF_TOKEN'] = hf_tok
        print('Loaded HF_TOKEN from Colab Secrets.')
    if GITHUB_TOKEN:
        print('Loaded GITHUB_TOKEN from Colab Secrets.')
except Exception:
    pass

if not GITHUB_TOKEN:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')

if GITHUB_TOKEN:
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/tothanhdat2006/MSSearch.git'
else:
    REPO_URL = 'https://github.com/tothanhdat2006/MSSearch.git'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git remote set-url origin {REPO_URL} && git checkout {BRANCH} && git pull origin {BRANCH}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
    sys.path.insert(0, os.path.join(REPO_DIR, 'experiments', 'baseline'))

In [ ]:
# Cell 3: Install Dependencies
!pip install --prefer-binary -r {REPO_DIR}/requirements.txt

In [ ]:
# Cell 4: Launch Evaluation
MODEL_NAME = 'Qwen/Qwen2.5-VL-7B-Instruct'  # Target VLM
POOL_MODE = 'N_5'                          # 'N_5', 'N_10', 'N_20', or 'crop'
EVAL_STRATEGY = 'multi_image'              # 'multi_image' or 'one_by_one'

START_IDX = None
END_IDX = None
MAX_SAMPLES = None
RESUME = False

cmd = [
    'python', f'{REPO_DIR}/experiments/baseline/eval_runner.py',
    '--model_path', f'"{MODEL_NAME}"',
    '--dataset_path', f'"{DATASET_PATH}"',
    '--image_dir', f'"{IMAGE_DIR}"',
    '--output_dir', f'"{OUTPUT_DIR}"',
    '--pool_mode', f'"{POOL_MODE}"',
    '--eval_strategy', f'"{EVAL_STRATEGY}"',
]

if RESUME: cmd.append('--resume')
if START_IDX is not None: cmd.extend(['--start_idx', str(START_IDX)])
if END_IDX is not None: cmd.extend(['--end_idx', str(END_IDX)])
if MAX_SAMPLES is not None: cmd.extend(['--max_samples', str(MAX_SAMPLES)])

full_cmd = ' '.join(cmd)
print(f'Executing: {full_cmd}')
!{full_cmd}